# Task 4: Game Title Classification (5-Class Image Classification — Macro $F_1$)

**Objective**: Classify gameplay screenshots into 5 game titles (`0: 7-11 Knights`, `1: FiveN`, `2: Free Fried`, `3: PUBE`, `4: Roblock`), evaluated by **Macro $F_1$ score**.

### Final Pipeline Overview
1. **Essential EDA & Validation Strategy** — Verify the ~4:1 class imbalance and widescreen aspect ratio (`224×416`), then construct **Scene-Grouped Stratified K-Fold** splits so entire near-duplicate gameplay scenes are held out together.
2. **Primary Backbone — LoRA Fine-Tuning of DINOv3 ViT-7B (75% Depth)**:
   - Truncate DINOv3 ViT-7B at block 30 (75% depth, which generalizes significantly better to unseen scenes than the final block 40) and attach rank-16 LoRA adapters (`alpha=32`) with a `LayerNorm + Linear` head over `[CLS, mean_patch]`.
   - Train two complementary depth-30 LoRA models (`blocks 20–29` and `blocks 10–29`), plus two multi-depth diversity models (`cut 24 / blocks 12–23` and `cut 40 / blocks 28–39`).
3. **Exact Weight-Space LoRA Model Soup + Dual-Head Refit**:
   - Merge the two depth-30 LoRA adapters exactly in weight space ($A_{\text{soup}} = [A_1; A_2]$, $B_{\text{soup}} = [B_1, B_2]/2 \implies B_{\text{soup}}A_{\text{soup}} = \frac{1}{2}(B_1A_1 + B_2A_2)$).
   - Refit two heads on frozen soup embeddings: a single-view (`plain`) head and a two-view (`plain + hflip`) head.
4. **Orthogonal Multi-Architecture Companions & 10-Backbone Joint Linear Fusion**:
   - Combine diverse fine-tuned vision families (ConvNeXt-Large/Base DINOv3, EVA02-Large/Base, DINOv3 ViT-Large, DINOv2-Large) with frozen foundation-model linear probes (multi-scale DINOv3 ViT-7B, C-RADIOv3-g, C-RADIOv3-H, C-RADIOv4-H, DINOv2-g, DINOv3 ViT-H+, Web-SSL 1B).
   - Train a convex L2-regularized **10-Backbone Early-Fusion Joint Linear Probe** (`56,576`-d L2-normalized embeddings, `wd=0.15`, zero hidden layers to prevent scene overfitting).
5. **OOF Temperature Calibration, Entropy-Weighted Adaptive Pooling & Train-Prior Adjustment**:
   - Calibrate each model's log-probabilities using its Scene-Fold-0 NLL-optimal temperature $T_m$ (correcting logit compression from label smoothing).
   - Fuse all components via **Calibrated Shannon Entropy-Weighted Pooling** (per-image certainty weighting — zero test-set statistics) and apply exact **uniform train-prior logit adjustment** ($\tau = 1/T_{\text{eff}}$).

> **Hardware Note**: Vision backbone fine-tuning (Stages 2–3) requires an **A100 GPU (40–80 GB)** and a `HF_TOKEN` environment variable (for gated DINOv3 weights). Stage 4 (Joint Linear Fusion) and Stage 5 (Calibration, Entropy-Weighted Ensembling & Inference) run cleanly on CPU.

In [1]:
import os
import glob
import copy
import math
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import roi_align
import timm
import timm.data

from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, log_loss
from scipy.optimize import minimize_scalar

warnings.filterwarnings("ignore")

def find_data_dir(subdir: str) -> str:
    for p in [os.path.join("..", "cpe342-kaggle-main", subdir), os.path.join("cpe342-kaggle-main", subdir)]:
        if os.path.isdir(p):
            return os.path.abspath(p)
    hits = glob.glob(f"/kaggle/input/**/{subdir}/train.csv", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError(f"Dataset not found for {subdir}. Expected in cpe342-kaggle-main/{subdir}.")

DATA_DIR = find_data_dir("04_image_classification")
WORK_DIR = "task4_artifacts"
OUT_CSV = "task4_predictions.csv"
os.makedirs(WORK_DIR, exist_ok=True)

CLASS_NAMES = ["7-11 Knights", "FiveN", "Free Fried", "PUBE", "Roblock"]
K = len(CLASS_NAMES)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tr = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
te = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
y = tr["label"].astype(int).values
train_prior = np.bincount(y, minlength=K) / len(y)

print(f"Device: {DEVICE} | Train: {len(tr):,} images | Test: {len(te):,} images")
print("Train class priors:", {CLASS_NAMES[k]: round(float(train_prior[k]), 4) for k in range(K)})

C:\Users\kiati\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu | Train: 31,546 images | Test: 25,889 images
Train class priors: {'7-11 Knights': 0.394, 'FiveN': 0.1397, 'Free Fried': 0.1774, 'PUBE': 0.1879, 'Roblock': 0.1011}


## 1. Essential EDA & Design Decisions

Quick verification of class imbalance and native widescreen resolution on training images:
- **Nearly 4:1 imbalance** (`39.4%` vs `10.1%`) under Macro $F_1$ $\rightarrow$ **Decision**: use $1/\sqrt{n_k}$ class weights during training and exact uniform train-prior adjustment ($P / \pi_{\text{train}}$) at inference.
- **Widescreen `~1.9:1` aspect ratio** $\rightarrow$ **Decision**: resize to `224×416` (preserving side HUD/gameplay details in multiples of ViT patch size 16).

In [2]:
sample_sizes = [Image.open(os.path.join(DATA_DIR, "train", fn)).size for fn in tr["file_name"].sample(200, random_state=42)]
ws = np.array([s[0] for s in sample_sizes])
hs = np.array([s[1] for s in sample_sizes])
print(f"Native train image resolution (200-image sample): median {np.median(ws):.0f}x{np.median(hs):.0f}, aspect ratio {np.median(ws/hs):.2f}")
print("-> Resizing to widescreen 224x416 preserves full gameplay framing in multiples of ViT patch size 16.")

Native train image resolution (200-image sample): median 230x120, aspect ratio 1.92
-> Resizing to widescreen 224x416 preserves full gameplay framing in multiples of ViT patch size 16.


## 2. Scene-Grouped Cross-Validation Split (Train-Only)

Training screenshots contain many near-duplicate frames from the same maps/scenes. To measure true generalization to unseen environments and obtain unbiased post-hoc calibration temperatures ($T_m$), we cluster training embeddings per class using k-means and construct **5-fold StratifiedGroupKFold** splits so entire scene clusters stay together.

In [3]:
def build_scene_folds(features: np.ndarray, labels: np.ndarray, clusters_per_class: int = 8, n_splits: int = 5, seed: int = 42) -> np.ndarray:
    """Cluster training embeddings per class and assign class-balanced scene-grouped folds."""
    X_norm = features.astype(np.float32)
    X_norm = X_norm / (np.linalg.norm(X_norm, axis=1, keepdims=True) + 1e-6)
    groups = np.zeros(len(labels), dtype=np.int32)
    gid = 0
    for k in range(K):
        idx = np.where(labels == k)[0]
        km = MiniBatchKMeans(n_clusters=clusters_per_class, random_state=seed + k, batch_size=2048, n_init=5)
        groups[idx] = gid + km.fit_predict(X_norm[idx])
        gid += clusters_per_class
    folds = np.zeros(len(labels), dtype=np.int8)
    sgkf = StratifiedGroupKFold(n_splits=n_splits)
    for f_id, (_, va_idx) in enumerate(sgkf.split(X_norm, labels, groups)):
        folds[va_idx] = f_id
    return folds

print("Scene-grouped CV helper ready (StratifiedGroupKFold over per-class scene clusters).")

Scene-grouped CV helper ready (StratifiedGroupKFold over per-class scene clusters).


## 3. Primary Model — LoRA Fine-Tuning of DINOv3 ViT-7B & Exact Weight-Space Model Soup

### Key Architectural Decisions
- **Truncation at 75% Depth (`CUT = 30` of 40 blocks)**: Intermediate representations at block 30 retain richer generalizable visual features than the final block 40 under unseen scene shifts.
- **LoRA Adapters (`rank = 16, alpha = 32`)**: Applied to all linear projections in the upper adapted blocks while keeping the 7B backbone frozen in `bfloat16` (only ~10–20M trainable fp32 parameters, fitting comfortably in a single A100 GPU with gradient checkpointing).
- **Multi-Depth Diversity**: Alongside two depth-30 adapters (`blocks 20–29` and `blocks 10–29`) merged via Model Soup, we also train a shallow-depth adapter (`CUT = 24, blocks 12–23`) and full-depth adapter (`CUT = 40, blocks 28–39`).
- **Exact LoRA Model Soup**: Because two independent LoRA runs learn unrelated low-rank subspaces, averaging $A$ and $B$ separately is invalid. Instead we concatenate $A_{\text{soup}} = [A_1; A_2]$ and $B_{\text{soup}} = [B_1, B_2] / 2$, reproducing the **exact** weight-space average $\Delta W_{\text{soup}} = \frac{1}{2}(B_1 A_1 + B_2 A_2)$, then refit two lightweight heads on frozen soup embeddings (single-view `plain` and two-view `plain + hflip`).

In [4]:
# ---------------- LoRA Module & GPU Image Pipeline ----------------
IMG_SIZE = (224, 416)  # (H, W), exact multiples of patch 16
LORA_RANK, LORA_ALPHA = 16, 32


class LoRALinear(nn.Module):
    """Drop-in fp32 LoRA wrapper around a frozen bf16 nn.Linear layer."""
    def __init__(self, base: nn.Linear, r: int = 16, alpha: int = 32):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        dev = base.weight.device
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=dev))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=dev))
        self.reset()

    def reset(self):
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x) + F.linear(F.linear(x, self.A.to(x.dtype)), self.B.to(x.dtype)) * self.scale


def preload_images_uint8(folder: str, filenames: np.ndarray, size_wh=(448, 240)) -> torch.Tensor:
    """Load RGB images in parallel into a compact uint8 tensor (N, H, W, 3)."""
    def _read(fn):
        return np.asarray(Image.open(os.path.join(folder, fn)).convert("RGB").resize(size_wh))
    with ThreadPoolExecutor(min(32, (os.cpu_count() or 4) * 4)) as ex:
        return torch.from_numpy(np.stack(list(ex.map(_read, filenames, chunksize=64))))


def gpu_augment(u8: torch.Tensor, out_size=IMG_SIZE) -> torch.Tensor:
    """Generic random resized crop + horizontal flip + photometric jitter on GPU."""
    dev = u8.device
    x = u8.permute(0, 3, 1, 2).float().div_(255.0)
    B, _, h, w = x.shape
    area = torch.empty(B, device=dev).uniform_(0.5, 1.0)
    ar = torch.exp(torch.empty(B, device=dev).uniform_(-0.15, 0.15))
    cw = (w * torch.sqrt(area * ar)).clamp(max=w)
    ch = (h * torch.sqrt(area / ar)).clamp(max=h)
    x0 = torch.rand(B, device=dev) * (w - cw)
    y0 = torch.rand(B, device=dev) * (h - ch)
    boxes = torch.stack([torch.arange(B, device=dev).float(), x0, y0, x0 + cw, y0 + ch], dim=1)
    x = roi_align(x, boxes, output_size=out_size, spatial_scale=1.0, sampling_ratio=2, aligned=True)
    flip = torch.rand(B, device=dev) < 0.5
    x[flip] = x[flip].flip(3)
    jit = lambda lo, hi: torch.empty(B, 1, 1, 1, device=dev).uniform_(lo, hi)
    x = x * jit(0.8, 1.2)
    m = x.mean((1, 2, 3), keepdim=True)
    x = (x - m) * jit(0.8, 1.2) + m
    g = x.mean(1, keepdim=True)
    x = (x - g) * jit(0.8, 1.2) + g
    return x.clamp_(0, 1)


def preprocess_view(u8: torch.Tensor, view: str = "plain", out_size=IMG_SIZE) -> torch.Tensor:
    x = u8.permute(0, 3, 1, 2).float().div_(255.0)
    x = F.interpolate(x, size=out_size, mode="bicubic", align_corners=False, antialias=True).clamp_(0, 1)
    return x.flip(3) if view == "hflip" else x


def merge_lora_soup_exact(lora_blocks: dict, checkpoints: list) -> dict:
    """Exact weight-space average of LoRA deltas: B_soup @ A_soup = mean_i(B_i @ A_i).
    checkpoints: list of (state_dict, lora_from_block) for cut-30 adapters.
    """
    w = 1.0 / len(checkpoints)
    n_per_blk = len(lora_blocks[29])
    collected, heads = {}, []
    for st, f_start in checkpoints:
        heads.append(st["head"])
        for idx, (a, b) in enumerate(st["lora"]):
            collected.setdefault((f_start + idx // n_per_blk, idx % n_per_blk), []).append((a.float(), b.float()))
    for blk_id, layers in lora_blocks.items():
        for j, layer in enumerate(layers):
            parts = collected.get((blk_id, j))
            if parts:
                layer.A = nn.Parameter(torch.cat([a for a, _ in parts], dim=0), requires_grad=False)
                layer.B = nn.Parameter(torch.cat([b for _, b in parts], dim=1) * w, requires_grad=False)
    avg_head = copy.deepcopy(heads[0])
    for k in avg_head:
        avg_head[k] = sum(h[k].float() for h in heads) / len(heads)
    return avg_head


def refit_soup_head(E_train: torch.Tensor, y_train: torch.Tensor, init_state: dict, epochs: int = 5, lr: float = 5e-4, seed: int = 42) -> nn.Module:
    """Refit LayerNorm+Linear head on frozen soup embeddings (warm-started from averaged head)."""
    torch.manual_seed(seed)
    D2 = E_train.shape[1]
    head = nn.Sequential(nn.LayerNorm(D2), nn.Dropout(0.1), nn.Linear(D2, K)).to(E_train.device)
    head.load_state_dict(init_state)
    cnt = torch.bincount(y_train, minlength=K).float()
    cw = (1.0 / torch.sqrt(cnt + 1.0))
    cw = (cw / cw.mean()).to(E_train.device)
    loss_fn = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=0.05)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    head.train()
    for _ in range(epochs):
        perm = torch.randperm(len(E_train), device=E_train.device)
        for i in range(0, len(E_train), 512):
            b = perm[i:i + 512]
            opt.zero_grad(set_to_none=True)
            loss_fn(head(E_train[b]), y_train[b]).backward()
            opt.step()
        sch.step()
    return head.eval()


def run_lora_7b_training(X_tr_u8: torch.Tensor, X_te_u8: torch.Tensor, y_tr: np.ndarray, scene_folds: np.ndarray = None):
    """Full GPU training routine for DINOv3 ViT-7B LoRA runs + exact Model Soup + Dual-Head Refit.
    If scene_folds is None, constructs 5-fold Scene-Grouped Stratified K-Fold splits directly from
    frozen DINOv3 ViT-7B embeddings (train-only) and saves all outputs under WORK_DIR.
    """
    assert torch.cuda.is_available(), "CUDA GPU (A100 40-80 GB recommended) required for ViT-7B LoRA fine-tuning."
    assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN environment variable (with DINOv3 license accepted)."

    model = timm.create_model("vit_7b_patch16_dinov3.lvd1689m", pretrained=True, num_classes=0, img_size=IMG_SIZE, dynamic_img_size=True)
    cfg = timm.data.resolve_model_data_config(model)
    all_blocks = list(model.blocks)
    model.norm = nn.Identity()
    model = model.to(torch.bfloat16).cuda().eval()
    for p in model.parameters():
        p.requires_grad_(False)

    n_prefix = model.num_prefix_tokens
    emb_dim = model.embed_dim
    mean = torch.tensor(cfg["mean"], device="cuda").view(1, 3, 1, 1)
    std = torch.tensor(cfg["std"], device="cuda").view(1, 3, 1, 1)

    def embed(x):
        tok = model.forward_features(((x - mean) / std).to(torch.bfloat16))
        return torch.cat([tok[:, 0], tok[:, n_prefix:].mean(1)], dim=1).float()

    @torch.no_grad()
    def embed_all(Xu8, view="plain", bs=64):
        model.eval()
        return torch.cat([embed(preprocess_view(Xu8[i:i + bs].cuda(non_blocking=True), view)).cpu() for i in range(0, len(Xu8), bs)])

    # Build train-only Scene-Grouped K-Fold split automatically from frozen depth-30 embeddings if not provided
    if scene_folds is None:
        model.blocks = nn.ModuleList(all_blocks[:30])
        E_frozen = embed_all(X_tr_u8, "plain").numpy()
        scene_folds = build_scene_folds(E_frozen, y_tr, clusters_per_class=8, n_splits=5, seed=42)
    np.save(os.path.join(WORK_DIR, "scene_folds.npy"), scene_folds)
    va0 = np.where(scene_folds == 0)[0]
    tr0 = np.where(scene_folds != 0)[0]
    np.save(os.path.join(WORK_DIR, "val0_idx.npy"), va0)

    # Wrap blocks 10..39 with LoRA adapters
    lora_map = {i: [] for i in range(10, 40)}
    for i in range(10, 40):
        for _, mod in list(all_blocks[i].named_modules()):
            for cn, child in list(mod.named_children()):
                if isinstance(child, nn.Linear):
                    wrapped = LoRALinear(child, LORA_RANK, LORA_ALPHA)
                    setattr(mod, cn, wrapped)
                    lora_map[i].append(wrapped)

    from timm.layers.config import set_reentrant_ckpt
    set_reentrant_ckpt(False)
    model.set_grad_checkpointing(True)

    head = nn.Sequential(nn.LayerNorm(2 * emb_dim), nn.Dropout(0.1), nn.Linear(2 * emb_dim, K)).cuda()
    Y_t = torch.tensor(y_tr, dtype=torch.long)

    @torch.no_grad()
    def predict_proba(Xu8, view="plain", bs=64):
        model.eval(); head.eval(); out = []
        for i in range(0, len(Xu8), bs):
            out.append(torch.softmax(head(embed(preprocess_view(Xu8[i:i + bs].cuda(non_blocking=True), view))), 1).cpu())
        return torch.cat(out).numpy()

    def train_one(tag, tr_idx, cut, lora_from, epochs=3, bs=32, seed=42):
        torch.manual_seed(seed); np.random.seed(seed)
        model.blocks = nn.ModuleList(all_blocks[:cut])
        for ls in lora_map.values():
            for l in ls:
                l.reset(); l.A.requires_grad_(False); l.B.requires_grad_(False)
        active = [l for i in range(lora_from, cut) for l in lora_map[i]]
        for l in active:
            l.A.requires_grad_(True); l.B.requires_grad_(True)
        for m in head.modules():
            if hasattr(m, "reset_parameters"):
                m.reset_parameters()

        lora_params = [p for l in active for p in (l.A, l.B)]
        opt = torch.optim.AdamW([
            {"params": lora_params, "lr": 1e-4, "weight_decay": 0.01},
            {"params": head.parameters(), "lr": 1e-3, "weight_decay": 0.05},
        ])
        spe = len(tr_idx) // bs
        total = spe * epochs
        warm = max(1, int(0.05 * total))
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1.0 + math.cos(math.pi * min(1.0, s / total)))
        )
        cnt = torch.bincount(Y_t[tr_idx], minlength=K).float()
        cw = (1.0 / torch.sqrt(cnt + 1.0)); cw = (cw / cw.mean()).cuda()
        loss_fn = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)

        for ep in range(epochs):
            model.train(); head.train()
            g = torch.Generator().manual_seed(seed + ep)
            perm = torch.from_numpy(tr_idx)[torch.randperm(len(tr_idx), generator=g)]
            for st in range(spe):
                b = perm[st * bs:(st + 1) * bs]
                loss = loss_fn(head(embed(gpu_augment(X_tr_u8[b].cuda(non_blocking=True)))), Y_t[b].cuda())
                opt.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(lora_params + list(head.parameters()), 1.0)
                opt.step(); sched.step()
        return {"lora": [(l.A.data.cpu().clone(), l.B.data.cpu().clone()) for l in active], "head": copy.deepcopy(head.state_dict())}

    # Train 4 complementary LoRA recipes on fold-0 split (for OOF calibration) + full train (for test)
    # 1. Depth-30 LoRA blocks 20-29  | 2. Depth-30 LoRA blocks 10-29  (merged into exact Model Soup)
    # 3. Depth-24 LoRA blocks 12-23  | 4. Depth-40 LoRA blocks 28-39  (multi-depth diversity)
    recipes = [("c30f20", 30, 20), ("c30f10", 30, 10), ("c24f12", 24, 12), ("c40f28", 40, 28)]
    all_idx = np.arange(len(y_tr))
    saved_states = {"val0": {}, "test": {}}

    for split_name, idx in [("val0", tr0), ("test", all_idx)]:
        target_X = X_tr_u8[va0] if split_name == "val0" else X_te_u8
        for tag, cut, lf in recipes:
            st = train_one(f"{tag}_{split_name}", idx, cut, lf)
            saved_states[split_name][tag] = st
            for view in ("plain", "hflip"):
                np.save(os.path.join(WORK_DIR, f"lora7b_{tag}_{split_name}_{view}.npy"), predict_proba(target_X, view))

        # Exact Model Soup of depth-30 adapters (c30f20 + c30f10) + Dual-Head Refit (1-view & 2-view)
        model.blocks = nn.ModuleList(all_blocks[:30])
        avg_head = merge_lora_soup_exact(
            {i: lora_map[i] for i in range(10, 30)},
            [(saved_states[split_name]["c30f20"], 20), (saved_states[split_name]["c30f10"], 10)],
        )
        E_p = embed_all(X_tr_u8[idx], "plain")
        E_h = embed_all(X_tr_u8[idx], "hflip")
        h1 = refit_soup_head(E_p.cuda(), Y_t[idx].cuda(), avg_head)
        h2 = refit_soup_head(torch.cat([E_p, E_h]).cuda(), torch.cat([Y_t[idx], Y_t[idx]]).cuda(), avg_head)

        for view in ("plain", "hflip"):
            E_eval = embed_all(target_X, view).cuda()
            with torch.no_grad():
                p1 = torch.softmax(h1(E_eval), 1).cpu().numpy()
                p2 = torch.softmax(h2(E_eval), 1).cpu().numpy()
            np.save(os.path.join(WORK_DIR, f"lora7b_soup2_1v_{split_name}_{view}.npy"), p1)
            np.save(os.path.join(WORK_DIR, f"lora7b_soup2_2v_{split_name}_{view}.npy"), p2)

print("Stage 3 (LoRA ViT-7B + Exact Weight-Space Model Soup + Dual-Head Refit) defined.")

Stage 3 (LoRA ViT-7B + Exact Weight-Space Model Soup + Dual-Head Refit) defined.


## 4. Joint 10-Backbone L2-Normalized Early-Fusion Linear Probe (Runs on CPU)

To complement the fine-tuned LoRA ViT-7B models without risking scene memorization, we concatenate frozen `[CLS, mean_patch]` embeddings across **10 diverse foundation-model views/backbones** (`DINOv3 ViT-7B [block 30, scale 1.5x, block 40, block 20]`, `C-RADIOv3-g`, `C-RADIOv3-H`, `C-RADIOv4-H`, `DINOv2-g`, `DINOv3 ViT-H+`, `Web-SSL 1B` — `56,576` total dimensions):
- Average `plain` and `hflip` embeddings per backbone, subtract the **training-only mean**, and L2-normalize each backbone block independently so every model family contributes on an equal scale.
- Train a strictly convex **L2-regularized linear classifier** (`AdamW`, `wd=0.15`, `1/sqrt(count)` class weights, `label_smoothing=0.05`, 2-seed full-train average). Having zero hidden layers prevents memorizing near-duplicate scenes while exploiting cross-family visual consensus.

In [5]:
def fit_joint_multibackbone_linear(
    X_tr_concat: np.ndarray,
    y_tr: np.ndarray,
    idx_fit: np.ndarray,
    X_eval_concat: np.ndarray,
    wd: float = 0.15,
    epochs: int = 30,
    lr: float = 1e-2,
    seeds=(42, 7),
) -> np.ndarray:
    """Convex L2-regularized linear probe over L2-normalized concatenated multi-backbone embeddings."""
    d = X_tr_concat.shape[1]
    Xa = torch.from_numpy(X_tr_concat[idx_fit])
    Ya = torch.from_numpy(y_tr[idx_fit]).long()
    cw = torch.tensor(1.0 / np.sqrt(np.bincount(y_tr[idx_fit], minlength=K)), dtype=torch.float32)
    cw = cw / cw.mean()
    loss_fn = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)

    preds = []
    for seed in seeds:
        torch.manual_seed(seed)
        clf = nn.Linear(d, K)
        opt = torch.optim.AdamW(clf.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        for _ in range(epochs):
            perm = torch.randperm(len(Xa))
            for i in range(0, len(Xa), 512):
                b = perm[i:i + 512]
                opt.zero_grad()
                loss_fn(clf(Xa[b].float()), Ya[b]).backward()
                opt.step()
            sch.step()
        clf.eval()
        chunks = []
        with torch.no_grad():
            for i in range(0, len(X_eval_concat), 4096):
                chunks.append(torch.softmax(clf(torch.from_numpy(X_eval_concat[i:i + 4096]).float()), 1).numpy())
        preds.append(np.concatenate(chunks, axis=0))
    return np.mean(preds, axis=0)

print("Stage 4 (10-Backbone Joint L2-Normalized Linear Probe) defined.")

Stage 4 (10-Backbone Joint L2-Normalized Linear Probe) defined.


## 5. OOF Temperature Calibration, Entropy-Weighted Adaptive Pooling & Exact Train-Prior Adjustment

### Why This Post-Processing Matters (All 100% Train-Only, Zero Test-Set Statistics)
1. **OOF Temperature Scaling**: Models trained with `label_smoothing = 0.1` (LoRA ViT-7B) output compressed log-probabilities (`T ≈ 0.38`), whereas unsmoothed models have wider logit ranges (`T ≈ 0.68`). Dividing each model's log-probabilities by its held-out Scene-Fold-0 NLL-optimal temperature $T_m$ restores true calibrated log-odds before pooling.
2. **Calibrated Shannon Entropy-Weighted Adaptive Pooling**: Instead of fixed weights on every image, each model's contribution on image $x$ is modulated by its calibrated certainty:
   $$w_m(x) = w_m^{\text{base}} \cdot \exp\!\left(-H(\tilde{p}_m^{\text{cal}}(x))\right)$$
   - On confident images, the Dual-Head LoRA Model Soup dominates (~90%+ effective weight).
   - On ambiguous out-of-distribution screenshots where LoRA has high entropy, the Multi-Depth LoRA (`cut 24`, `cut 40`), Multi-Architecture Base (`ConvNeXt-L/B`, `EVA02-L/B`, `DINOv3 ViT-L`, `DINOv2-L`, `C-RADIOv3-g`, etc.), and 10-Backbone Joint Linear Probe step in automatically to resolve ties.
3. **Exact Uniform Train-Prior Logit Adjustment**: Subtracting $\left(\sum_m \frac{w_m(x)}{T_m}\right) \log \pi_{\text{train}}$ converts the calibrated ensemble posterior to the exact Bayes-optimal **uniform-prior** distribution for Macro $F_1$, using only training class counts.

In [6]:
lg = lambda p: np.log(np.clip(p, 1e-9, 1.0))
sm = lambda z: np.exp(z - z.max(1, keepdims=True)) / np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True)
ent = lambda p: -(p * np.log(np.clip(p, 1e-12, 1.0))).sum(1)


def fit_temperature(P_val: np.ndarray, y_val: np.ndarray) -> float:
    """Fit scalar post-hoc temperature T minimizing NLL strictly on held-out scene validation fold."""
    L = lg(P_val)
    return float(minimize_scalar(lambda T: log_loss(y_val, sm(L / T)), bounds=(0.1, 3.0), method="bounded").x)


def entropy_weighted_calibrated_pool(
    components: list,
    prior: np.ndarray,
    gamma: float = 1.0,
) -> np.ndarray:
    """Calibrated Shannon Entropy-Weighted Pooling with exact uniform train-prior adjustment.

    components: list of (P_proba, T_oof, base_weight)
    prior: training class prior vector (length K), zero test-set statistics used.
    """
    num = np.zeros_like(components[0][0], dtype=np.float64)
    t_scale = np.zeros((len(components[0][0]), 1), dtype=np.float64)
    for P, T_m, w_base in components:
        L_cal = lg(P) / T_m
        P_cal = sm(L_cal)
        w_img = w_base * np.exp(-gamma * ent(P_cal))[:, None]
        num += w_img * L_cal
        t_scale += w_img / T_m
    # Exact uniform target prior adjustment in each sample's local calibrated logit scale
    return sm(num - t_scale * np.log(prior))


# Final Ensemble Component Weights (validated on Scene-Grouped Fold 0: 0.99482 Macro F1, 21/5,113 errors)
FINAL_WEIGHTS = {
    "lora7b_soup2_1v": 0.52,  # Exact LoRA Model Soup (cut 30: blocks 20-29 + 10-29), single-view refit head
    "lora7b_soup2_2v": 0.25,  # Exact LoRA Model Soup (cut 30: blocks 20-29 + 10-29), two-view refit head
    "lora7b_c24f12":   0.04,  # Multi-depth LoRA diversity (60% depth, blocks 12-23)
    "lora7b_c40f28":   0.04,  # Multi-depth LoRA diversity (100% depth, blocks 28-39)
    "multi_arch_base": 0.09,  # Multi-architecture FT (ViT-L, EVA02-L/B, ConvNeXt-L/B, DINOv2-L) + frozen heads (incl. C-RADIOv3-g)
    "joint_10mb_lin":  0.06,  # 10-Backbone L2-Normalized Joint Linear Probe (56,576-d, wd=0.15)
}
assert abs(sum(FINAL_WEIGHTS.values()) - 1.0) < 1e-9

print("Final ensemble weights:", FINAL_WEIGHTS)

Final ensemble weights: {'lora7b_soup2_1v': 0.52, 'lora7b_soup2_2v': 0.25, 'lora7b_c24f12': 0.04, 'lora7b_c40f28': 0.04, 'multi_arch_base': 0.09, 'joint_10mb_lin': 0.06}


## 6. End-to-End Inference & Save Final Predictions (`task4_predictions.csv`)

- **Fresh End-to-End GPU Run (`RUN_FULL_GPU_TRAINING = True`)**: Automatically builds train-only `scene_folds.npy` from frozen DINOv3 ViT-7B embeddings, trains all 4 LoRA adapters + Exact Weight-Space Model Soup + Dual-Head Refit into `WORK_DIR`, and immediately executes Stage 5 below to save `task4_predictions.csv` (25,889 rows aligned to `test.csv`).
- **Modular / Precomputed Run (`RUN_FULL_GPU_TRAINING = False`)**: Loads any probability arrays present in `WORK_DIR` (the 4 primary LoRA+Soup models, plus optional companion `multi_arch_base` and `joint_10mb_lin` arrays trained across multi-architecture sessions), re-normalizes `FINAL_WEIGHTS` over all available components, calibrates temperatures on Scene Fold 0, and saves `task4_predictions.csv`.

In [7]:
RUN_FULL_GPU_TRAINING = False  # Set True on an A100 GPU with HF_TOKEN to run Stages 2-3 from scratch

if RUN_FULL_GPU_TRAINING:
    X_tr_u8 = preload_images_uint8(os.path.join(DATA_DIR, "train"), tr["file_name"].values)
    X_te_u8 = preload_images_uint8(os.path.join(DATA_DIR, "test"), te["file_name"].values)
    run_lora_7b_training(X_tr_u8, X_te_u8, y, scene_folds=None)


def load_tta_geo(prefix: str) -> np.ndarray:
    """Symmetric 50/50 log-probability average of plain and horizontal-flip views."""
    return sm(0.5 * lg(np.load(f"{prefix}_plain.npy")) + 0.5 * lg(np.load(f"{prefix}_hflip.npy")))


def load_workdir_components(work_dir: str):
    """Load OOF (Scene Fold 0) and Test probability arrays from WORK_DIR."""
    val0_path = os.path.join(work_dir, "val0_idx.npy")
    folds_path = os.path.join(work_dir, "scene_folds.npy")
    if os.path.exists(val0_path):
        va0 = np.load(val0_path)
    elif os.path.exists(folds_path):
        va0 = np.where(np.load(folds_path) == 0)[0]
    else:
        return None

    oof_dict, test_dict = {}, {}
    for name in FINAL_WEIGHTS:
        # Check for 2-view TTA files (plain + hflip) produced by Stage 3
        p_oof = os.path.join(work_dir, f"{name}_val0")
        p_te = os.path.join(work_dir, f"{name}_test")
        if os.path.exists(f"{p_oof}_plain.npy") and os.path.exists(f"{p_te}_plain.npy"):
            oof_dict[name] = load_tta_geo(p_oof)
            test_dict[name] = load_tta_geo(p_te)
        # Or single-array pre-pooled probability files (e.g., Stage 4 Joint Linear / Multi-Arch Base)
        elif os.path.exists(f"{p_oof}.npy") and os.path.exists(f"{p_te}.npy"):
            oof_dict[name] = np.load(f"{p_oof}.npy")
            test_dict[name] = np.load(f"{p_te}.npy")

    if not oof_dict:
        return None
    return oof_dict, test_dict, y[va0]


bundle = load_workdir_components(WORK_DIR)
if bundle is not None:
    oof_dict, test_dict, y_val0 = bundle
    active_keys = [k for k in FINAL_WEIGHTS if k in oof_dict]
    w_sum = sum(FINAL_WEIGHTS[k] for k in active_keys)
    active_weights = {k: FINAL_WEIGHTS[k] / w_sum for k in active_keys}

    temps = {k: fit_temperature(oof_dict[k], y_val0) for k in active_keys}
    print(f"Loaded {len(active_keys)} / {len(FINAL_WEIGHTS)} ensemble components from {WORK_DIR}/")
    print("Scene-Fold-0 NLL-optimal temperatures:", {k: round(v, 3) for k, v in temps.items()})

    oof_ens = entropy_weighted_calibrated_pool([(oof_dict[k], temps[k], active_weights[k]) for k in active_keys], train_prior, gamma=1.0)
    val_pred = oof_ens.argmax(axis=1)
    print(f"Scene-Fold-0 Ensemble Macro F1: {f1_score(y_val0, val_pred, average='macro'):.5f} ({int((val_pred != y_val0).sum())} / {len(y_val0)} errors)")

    test_ens = entropy_weighted_calibrated_pool([(test_dict[k], temps[k], active_weights[k]) for k in active_keys], train_prior, gamma=1.0)
    pred_test = test_ens.argmax(axis=1)

    sub_t4 = pd.DataFrame({"id": te["id"].values, "pred": pred_test})
    sub_t4.to_csv(OUT_CSV, index=False, lineterminator="\n")
    print(f"Saved {OUT_CSV} | rows={len(sub_t4):,}")
    print("Predicted test class share:", {CLASS_NAMES[k]: round(float((pred_test == k).mean()), 4) for k in range(K)})
    display(sub_t4.head())
else:
    print(f"No trained probability arrays found in {WORK_DIR}/.")
    print("Set RUN_FULL_GPU_TRAINING = True on an A100 GPU with HF_TOKEN to train from scratch and generate task4_predictions.csv.")

Loaded 6 / 6 ensemble components from task4_artifacts/
Scene-Fold-0 NLL-optimal temperatures: {'lora7b_soup2_1v': 0.38, 'lora7b_soup2_2v': 0.378, 'lora7b_c24f12': 0.426, 'lora7b_c40f28': 0.524, 'multi_arch_base': 0.677, 'joint_10mb_lin': 0.454}
Scene-Fold-0 Ensemble Macro F1: 0.99482 (21 / 5113 errors)
Saved task4_predictions.csv | rows=25,889
Predicted test class share: {'7-11 Knights': 0.1287, 'FiveN': 0.284, 'Free Fried': 0.1466, 'PUBE': 0.234, 'Roblock': 0.2067}


,id,pred
0,ANS00001,1
1,ANS00002,2
2,ANS00003,3
3,ANS00004,0
4,ANS00005,3
